# 02. Previsão de Risco de Embargos

**Objetivo:** Prever municípios com maior risco de embargos ambientais para gestão de risco em cadeias de suprimento.

**Impacto no Negócio:** Implementar due diligence ambiental para fornecedores em municípios de alto risco de compliance.

**Dados de Entrada:** `data/04_modelagem/dataset_preditivo_com_precos.parquet`

**Dados de Saída:** `data/03_gold/ranking_risco_embargos_2023.parquet`

In [ ]:
# ============================================================================
# MONTAR GOOGLE DRIVE (APENAS COLAB)
# ============================================================================

def montar_google_drive():
    """Monta o Google Drive no Colab."""
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✓ Google Drive montado em /content/drive")
        return True
    except Exception as e:
        print(f"⚠️  Erro ao montar Google Drive: {e}")
        return False

# Detecta se está no Colab e tenta montar o Drive
try:
    import google.colab
    print("📤 Ambiente Google Colab detectado")
    print("Montando Google Drive...")
    montar_google_drive()
except ImportError:
    print("✓ Ambiente local detectado - não é necessário montar Drive")

In [ ]:
# ============================================================================
# CONFIGURAÇÃO DE AMBIENTE
# ============================================================================

import sys
import os
from pathlib import Path

# Detectar ambiente e configurar caminho corretamente
try:
    import google.colab
    print("📤 Ambiente Google Colab detectado")
    # No Colab, usar o diretório do drive
    if os.path.exists('/content/drive/MyDrive/dados_analise'):
        os.chdir('/content/drive/MyDrive/dados_analise')
        print("✓ Diretório alterado para: /content/drive/MyDrive/dados_analise")
    else:
        print("⚠️  Diretório dados_analise não encontrado no Drive")
except ImportError:
    print("✓ Ambiente local detectado")
    # Local, usar diretório atual
    current_dir = Path.cwd()
    # Se estiver em notebooks_analise_preditiva, voltar para o root
    if 'notebooks_analise_preditiva' in str(current_dir):
        os.chdir(current_dir.parent)
        print(f"✓ Diretório alterado para: {current_dir.parent}")

print(f"✓ Diretório de trabalho atual: {os.getcwd()}")

# ============================================================================
# CONFIGURAÇÃO DE CAMINHOS
# ============================================================================

# Caminho direto para os dados (já está no drive)
CAMINHO_DADOS = 'data/04_modelagem/dataset_preditivo_com_precos.parquet'
CAMINHO_SAIDA = 'data/03_gold/ranking_risco_embargos_2023.parquet'

print(f"\nCaminho dos dados: {CAMINHO_DADOS}")
print(f"Caminho de saída: {CAMINHO_SAIDA}")

# Verificar se o arquivo existe
if os.path.exists(CAMINHO_DADOS):
    print(f"✓ Arquivo de dados encontrado")
else:
    print(f"⚠️  Arquivo de dados não encontrado: {CAMINHO_DADOS}")

In [ ]:
## 1. Configuração e Importações
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURAÇÃO
# ============================================================================

# Features selecionadas para o modelo (focadas em compliance)
FEATURES_MODELO_EMBARGOS = [
    'cod_ibge', 'ano', 'vab_agro_mil_reais', 'ppm_bovinos_cabecas', 
    'area_desmatada_ha', 'idhm', 'precipitacao_total_mm', 'anos_obs',
    'log_bovinos', 'log_vab', 'pressao_economica',
    'preco_boi_gordo_rs', 'preco_milho_rs', 'preco_soja_rs',
    'producao_soja_mil_ton', 'producao_milho_mil_ton', 'pressao_agro_alta',
    'indice_pressao_preco'
]

TARGET_EMBARGOS = 'tem_embargos'

# UFs da Amazônia Legal
UFS_AMAZONIA_LEGAL = ['AC', 'AM', 'AP', 'MA', 'MT', 'PA', 'RO', 'RR', 'TO']

# Anos para divisão temporal
ANO_LIMITE_TREINO = 2022
ANO_TESTE = 2023
ANO_PREVISAO = 2023

In [ ]:
# Célula removida - código movido para célula 9

## 3. Preparação de Features

In [ ]:
# Features para modelo de embargos (focadas em compliance)
features_embargos = [
    'cod_ibge', 'ano', 'vab_agro_mil_reais', 'ppm_bovinos_cabecas', 
    'area_desmatada_ha', 'idhm', 'precipitacao_total_mm', 'anos_obs',
    'log_bovinos', 'log_vab', 'pressao_economica',
    'preco_boi_gordo_rs', 'preco_milho_rs', 'preco_soja_rs',
    'producao_soja_mil_ton', 'producao_milho_mil_ton', 'pressao_agro_alta',
    'indice_pressao_preco'
]

# Preparar dataset para modelo
df_modelo_embargos = df_amazonia[features_embargos + ['tem_embargos']].copy()
df_modelo_embargos = df_modelo_embargos.dropna()

print(f'Dataset para modelo de embargos: {df_modelo_embargos.shape[0]:,} observações')
print(f'Features: {len(features_embargos)}')

## 4. Divisão Temporal de Dados

In [ ]:
# Divisão temporal (treinar com dados passados, testar com dados futuros)
ANO_LIMITE_TREINO = 2022
ANO_TESTE = 2023

X_train = X[X['ano'] <= ANO_LIMITE_TREINO]
X_test = X[X['ano'] == ANO_TESTE]
y_train = y[X['ano'] <= ANO_LIMITE_TREINO]
y_test = y[X['ano'] == ANO_TESTE]

print(f'Treino: {X_train.shape[0]:,} observações ({X_train["ano"].min()}-{X_train["ano"].max()})')
print(f'Teste: {X_test.shape[0]:,} observações ({X_test["ano"].min()}-{X_test["ano"].max()})')
print(f'\nDistribuição target treino: {y_train.mean()*100:.2f}% positivos')
print(f'Distribuição target teste: {y_test.mean()*100:.2f}% positivos')

## 5. Treinamento do Modelo

In [ ]:
# Calcular pesos das classes
class_weights_emb = compute_class_weight('balanced', classes=np.unique(y_train_emb), y=y_train_emb)
class_weight_dict_emb = {0: class_weights_emb[0], 1: class_weights_emb[1]}
print(f'Class weights: {class_weight_dict_emb}')

# Treinar modelo Random Forest para embargos
rf_embargos = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_split=50,
    min_samples_leaf=20,
    class_weight=class_weight_dict_emb,
    random_state=42,
    n_jobs=-1,
    max_features='sqrt'
)

rf_embargos.fit(X_train_emb, y_train_emb)
print('\nModelo de embargos treinado com sucesso!')

## 6. Avaliação do Modelo

In [ ]:
# Avaliar modelo
y_pred_proba_emb = rf_embargos.predict_proba(X_test_emb)[:, 1]

# Otimizar threshold
precision_emb, recall_emb, thresholds_emb = precision_recall_curve(y_test_emb, y_pred_proba_emb)
f1_scores_emb = 2 * (precision_emb * recall_emb) / (precision_emb + recall_emb + 1e-8)
best_threshold_emb = thresholds_emb[np.argmax(f1_scores_emb)]

y_pred_optimized_emb = (y_pred_proba_emb >= best_threshold_emb).astype(int)

print('=== AVALIAÇÃO DO MODELO DE EMBARGOS ===')
print(f'ROC-AUC: {roc_auc_score(y_test_emb, y_pred_proba_emb):.4f}')
print(f'Precision-Recall AUC: {auc(recall_emb, precision_emb):.4f}')
print(f'Threshold utilizado: {best_threshold_emb:.4f}')

print('\nRelatório de Classificação:')
print(classification_report(y_test_emb, y_pred_optimized_emb, target_names=['Sem Embargos', 'Com Embargos']))

## 7. Feature Importance

In [ ]:
# Feature importance para embargos
feature_importance_emb = pd.DataFrame({
    'feature': X_train_emb.columns,
    'importance': rf_embargos.feature_importances_
}).sort_values('importance', ascending=False)

print('\n=== TOP 10 FEATURES MAIS IMPORTANTES PARA EMBARGOS ===')
print(feature_importance_emb.head(10).to_string(index=False))

## 8. Previsão para 2023 e Ranking de Risco

In [ ]:
# Criar ranking de municípios em risco de embargos para 2023
df_2022 = df_amazonia[df_amazonia['ano'] == ANO_TESTE].copy()
X_2023_emb = df_2022[features_embargos].select_dtypes(include=[np.number])

df_2022['probabilidade_embargos_2023'] = rf_embargos.predict_proba(X_2023_emb)[:, 1]

# Remover duplicatas
ranking_embargos = df_2022[['cod_ibge', 'municipio', 'uf', 'probabilidade_embargos_2023',
                            'area_embargada_ha', 'vab_agro_mil_reais']].drop_duplicates('cod_ibge')
ranking_embargos = ranking_embargos.sort_values('probabilidade_embargos_2023', ascending=False)

print(f'\n=== TOP 20 MUNICÍPIOS COM MAIOR PROBABILIDADE DE EMBARGOS EM 2023 ===')
print(ranking_embargos.head(20).to_string(index=False))

## 9. Estatísticas de Impacto

In [ ]:
# Estatísticas de impacto
top_50_emb = ranking_embargos.head(50)
print('\n=== ESTATÍSTICAS DE IMPACTO DO RANKING DE EMBARGOS ===')
print(f'Top 50 municípios - Probabilidade média: {top_50_emb["probabilidade_embargos_2023"].mean()*100:.1f}%')
print(f'Top 50 municípios - Área embargada histórica: {top_50_emb["area_embargada_ha"].sum():,.0f} ha')
print(f'Top 50 municípios - VAB agropecuário: R$ {top_50_emb["vab_agro_mil_reais"].sum()*1000:,.0f}')

## 10. Salvamento dos Resultados

In [ ]:
# Salvar ranking
ranking_embargos.to_parquet(CAMINHO_SAIDA, index=False)
print(f'\nRanking de embargos salvo em {CAMINHO_SAIDA}')
print(f'Total de municípios: {len(ranking_embargos)}')

## 11. Conclusão

**Resumo da Análise:**
- Modelo de Random Forest treinado com ROC-AUC de {roc_auc_score(y_test_emb, y_pred_proba_emb):.4f}
- {len(ranking_embargos)} municípios classificados por probabilidade de embargos
- Top 50 municípios identificados para due diligence ambiental

**Impacto no Negócio:**
- Gestão de risco em cadeias de suprimento
- Due diligence ambiental para fornecedores
- Base para sistema de compliance

**Próximos Passos:**
- Implementar verificação de certificações ambientais
- Criar sistema de rastreamento de cadeia de suprimento
- Desenvolver lista de exclusão para fornecedores de alto risco